# Cross-task remapping + coherent rotation (LEC)

El-Gaby et al. 2024 (Figure 3) reimplemented on the LEC data via [`remapping_rotation_analysis.py`](remapping_rotation_analysis.py): cross-task remapping of task-state (goal-progress) tuning, coherent rotation of neuron pairs, and the module-clustering / silhouette analysis.

**Setup chain**: imports → folder paths → pickle-load `data_dic` → build `mouse_recdays` and `valid_sessions_dic` → run the remapping/rotation analysis. (The setup cells are shared with the elastic-net notebooks; the extra `norm_neurons_dic`/`session_inds_dic`/`tasks_dic` loads are harmless and unused here.)

In [8]:
import numpy as np
import scipy.stats as st
from scipy import stats
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, pickle
from tqdm import tqdm


In [9]:
Data_folder="../data/processed_data"
figures_folder="../data/figures"
neuron_folder = f"{Data_folder}/neuron_raw_mingyutest"
trialtimes_folder = f"{Data_folder}/trialtimes_raw_mingyutest"
tracking_folder = f"{Data_folder}/processed"


In [10]:
import pickle
import os

# Define the folder where the processed data is saved
processed_data_folder = "../data/processed_data"

# Load the main data dictionary
with open(os.path.join(processed_data_folder, 'data_dic_for_yaren.pkl'), 'rb') as f:
    data_dic = pickle.load(f)

# Load the normalized neurons dictionary
with open(os.path.join(processed_data_folder, 'norm_neurons_dic.pkl'), 'rb') as f:
    norm_neurons_dic = pickle.load(f)

# Load the session indices dictionary
with open(os.path.join(processed_data_folder, 'session_inds_dic.pkl'), 'rb') as f:
    session_inds_dic = pickle.load(f)

# Load the tasks dictionary
with open(os.path.join(processed_data_folder, 'tasks_dic.pkl'), 'rb') as f:
    tasks_dic = pickle.load(f)

print(f"Loaded data dictionaries from {processed_data_folder}")


Loaded data dictionaries from ../data/processed_data


In [11]:
all_files = os.listdir(neuron_folder)
neuron_files = [f for f in all_files if f.startswith('Neuron_raw_') and f.endswith('.npy')]

mouse_recdays_ = []
for f in neuron_files:
    # Remove prefix 'Neuron_raw_' and suffix '.npy'
    stripped_name = f[len('Neuron_raw_'):-len('.npy')]
    # Split by '_' and rejoin all parts except the last one (session number)
    mouse_recday = '_'.join(stripped_name.split('_')[:-1])
    mouse_recdays_.append(mouse_recday)

# Find the unique mouse_recday identifiers
mouse_recdays = np.unique(mouse_recdays_)

# Filter out identifiers containing '_sb'
unique_mouse_recdays = [recday for recday in mouse_recdays if '_sb' not in recday]

print("Unique mouse_recday identifiers (without '_sb'):")
print(unique_mouse_recdays)

# Assign the first unique identifier to mouse_recday for later use
if len(unique_mouse_recdays) > 0:
    mouse_recday = unique_mouse_recdays[0]

mouse_recdays = unique_mouse_recdays


Unique mouse_recday identifiers (without '_sb'):
[np.str_('ah08_20250613_20250615'), np.str_('ah08_20250616_20250617'), np.str_('ah08_20250618_20250619'), np.str_('ah08_20250620_20250623'), np.str_('ah08_20250624_20250625'), np.str_('ah10_20250613_20250615'), np.str_('ah10_20250616_20250617'), np.str_('ah10_20250618_20250619'), np.str_('ah10_20250620_20250623'), np.str_('ah10_20250624_20250625'), np.str_('ly05_20250613_20250615'), np.str_('ly05_20250616_20250617'), np.str_('ly05_20250618_20250619'), np.str_('ly05_20250624_20250625'), np.str_('ly06_20250613_20250615'), np.str_('ly06_20250616_20250617'), np.str_('ly06_20250618_20250619'), np.str_('ly06_20250620_20250623'), np.str_('ly06_20250624_20250625'), np.str_('ly07_20250613_20250615'), np.str_('ly07_20250616_20250617'), np.str_('ly07_20250618_20250619'), np.str_('ly07_20250620_20250623'), np.str_('ly07_20250624_20250625')]


In [12]:
# Build valid_sessions_dic: for each mouse_recday, keep one session per unique task structure
valid_sessions_dic = {}
for mouse_recday in mouse_recdays:
    valid_sessions = []
    tasks = []
    for session in list(data_dic[mouse_recday].keys()):
        if session == 'valid_sessions':
            continue
        if data_dic[mouse_recday][session]['num_trials'] < 5:
            print(f'{mouse_recday} session {session}: not enough trials, skipping')
            continue
        if 'defaultdict' in str(data_dic[mouse_recday][session]['Task']):
            print(f'{mouse_recday} session {session}: no task, skipping')
            continue
        if not any(np.array_equal(data_dic[mouse_recday][session]['Task'], candidate) for candidate in tasks):
            tasks.append(data_dic[mouse_recday][session]['Task'])
            valid_sessions.append(session)
    print(f'{mouse_recday}: {valid_sessions}')
    valid_sessions_dic[mouse_recday] = valid_sessions


ah08_20250613_20250615 session 7: not enough trials, skipping
ah08_20250613_20250615: [0, 1, 2, 4, 5, 6]
ah08_20250616_20250617 session 2: not enough trials, skipping
ah08_20250616_20250617 session 7: not enough trials, skipping
ah08_20250616_20250617: [0, 1, 4, 5, 6]
ah08_20250618_20250619 session 7: not enough trials, skipping
ah08_20250618_20250619: [0, 1, 2, 4, 5, 6]
ah08_20250620_20250623 session 3: not enough trials, skipping
ah08_20250620_20250623 session 6: not enough trials, skipping
ah08_20250620_20250623 session 7: not enough trials, skipping
ah08_20250620_20250623: [0, 1, 2, 4, 5]
ah08_20250624_20250625 session 1: not enough trials, skipping
ah08_20250624_20250625 session 3: not enough trials, skipping
ah08_20250624_20250625 session 4: not enough trials, skipping
ah08_20250624_20250625 session 5: not enough trials, skipping
ah08_20250624_20250625 session 8: not enough trials, skipping
ah08_20250624_20250625 session 9: not enough trials, skipping
ah08_20250624_20250625 sessi

# Cross-task remapping + coherent rotation (El-Gaby Figure3 reimplementation)

Uses `remapping_rotation_analysis.py`. For each recday it builds per-neuron, per-task 360-bin
goal-progress tuning curves (matched cells across tasks), then:
 * **remapping** — single-cell rotation offset vs a reference task (peak at 0 = generalises);
 * **coherent rotation** — whether neuron *pairs* keep their relative rotation across tasks
   (population rotates as a rigid body), vs a 1/num_states chance level;
 * **X-vs-X′** — the same on repeated identical-task sessions (no-remapping baseline).

Neurons are included if state-tuned in ≥half of their tasks. Figures saved to a separate
`remapping_rotation_<timestamp>` directory; v2/v3 outputs untouched.


In [13]:
exec(open('remapping_rotation_analysis.py').read())
from datetime import datetime

config_rr = RemapConfig()   # num_states=4, sigma=10, coherence threshold 45 deg, tuned in >=half tasks
save_dir_rr = f'../data/figures/remapping_rotation_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

results_rr, summary_rr = run_remapping_rotation_all_mice(
    data_dic, valid_sessions_dic, config=config_rr, save_dir=save_dir_rr, verbose=True)

print('\nSUMMARY:')
for k, v in summary_rr.items():
    print(f'  {k}: {v}')


Processing ah08_20250613_20250615 ...
  ah08_20250613_20250615: 6 tasks, 36 included neurons, coherent_prop=0.081, X-vs-X' present
Processing ah08_20250616_20250617 ...
  ah08_20250616_20250617: 5 tasks, 38 included neurons, coherent_prop=0.064, X-vs-X' present
Processing ah08_20250618_20250619 ...
  ah08_20250618_20250619: 6 tasks, 65 included neurons, coherent_prop=0.112, X-vs-X' present
Processing ah08_20250620_20250623 ...
  ah08_20250620_20250623: 5 tasks, 41 included neurons, coherent_prop=0.094
Processing ah08_20250624_20250625 ...
  ah08_20250624_20250625: 4 tasks, 70 included neurons, coherent_prop=0.080
Processing ah10_20250613_20250615 ...
  ah10_20250613_20250615: 6 tasks, 116 included neurons, coherent_prop=0.156, X-vs-X' present
Processing ah10_20250616_20250617 ...
  ah10_20250616_20250617: 6 tasks, 126 included neurons, coherent_prop=0.169, X-vs-X' present
Processing ah10_20250618_20250619 ...
  ah10_20250618_20250619: 6 tasks, 139 included neurons, coherent_prop=0.121,

In [14]:
# Single recday (quick inspection)
mr = [m for m in data_dic if len(valid_sessions_dic.get(m, [])) >= 3][0]
res = analyse_recday(data_dic, mr, valid_sessions_dic[mr], config_rr, verbose=True)
print(mr, 'included neurons:', res['n_included'],
      '| coherent_prop:', res.get('coherent_prop'))


  ah08_20250613_20250615: 6 tasks, 36 included neurons, coherent_prop=0.081, X-vs-X' present
ah08_20250613_20250615 included neurons: 36 | coherent_prop: 0.08095238095238096
